In [ ]:
import os
import pandas as pd

raw_dir = "/Users/muna/Hana_research/data/raw/AllKarte"

# ファイル一覧を確認
csv_files = [f for f in os.listdir(raw_dir) if f.endswith('.csv')]
print(f"CSVファイル数: {len(csv_files)}")
for f in csv_files:
    print(f)

CSVファイル数: 33
2024-1.csv
2019-1.csv
2024-2.csv
2022-4.csv
2019-2.csv
2024-3.csv
2022-1.csv
2020-3.csv
2020-2.csv
2024-6.csv
2024-4.csv
2022-2.csv
2020-1.csv
2022-3.csv
2024-5.csv
2023-4.csv
2018-2.csv
2025-2.csv
2025-3.csv
2018-1.csv
2025-1.csv
2021-4.csv
2023-2.csv
2025-4.csv
2023-3.csv
2021-1.csv
2021-3.csv
2023-1.csv
2021-2.csv
2015.csv
2016.csv
2017-1.csv
2017-2.csv


In [6]:
# 各ファイルの先頭を確認（cp932固定）
for f in csv_files[:3]:
    path = os.path.join(raw_dir, f)
    df = pd.read_csv(path, encoding='cp932', nrows=5)
    print(f"\n=== {f} ===")
    print(df.head())
    print(df.dtypes)


=== 2024-1.csv ===
   NO 診療タイプ                診療日時  フラグ備考      ID   患者氏名        患者施設  \
0   1  深夜往診  2024/5/31(金) 22:15    NaN  230369  冨田 節子         NaN   
1   2  電話再診  2024/5/31(金) 19:58    NaN  230369  冨田 節子         NaN   
2   3   その他  2024/5/31(金) 19:00    NaN  213217  原田 繁和          外来   
3   4   その他  2024/5/31(金) 19:00    NaN  211795  古川 正一          外来   
4   5   その他  2024/5/31(金) 19:00    NaN  210224  澤田 林二  なごみの郷　すずらん   

                                               カルテ内容   医師名  \
0  ■往診理由\r\n口唇ヘルペス\r\n\r\n■本人及び家族からの訴え\r\n夜になり、口唇...   助川誠   
1  ■電話内容\r\n娘様より電話連絡\r\n\r\n今日、定期往診来てもらい、口内炎の薬出して...   助川誠   
2  所見】\r\n胸部単純CT\r\n2023年6月5日のCTと比較しました。\r\n右肺中葉や...  高橋直人   
3  所見】\r\n胸部単純CT\r\n左肺下葉に4mm程度の小結節が認められます（図1）。炎症後...  高橋直人   
4  所見】\r\n胸部～骨盤部単純CT\r\n両肺に活動性炎症や腫瘤性病変はありません。\r\n...  山本直史   

                 終了時間  ...  身長  体重                 更新日時           入力者ID 看護師氏名  \
0  2024/5/31(金) 22:25  ... NaN NaN  2024-06-05 16:30:12   20131munakata  鎌田小雪   
1       2024/5/31(金)   ... NaN

In [1]:
import sqlite3
import pandas as pd
import re
from collections import defaultdict
from pathlib import Path

# =========================
# DB接続
# =========================
db_path = "/Users/muna/Hana_research/data/db/Hana_Research.db"

conn = sqlite3.connect(db_path)

# =========================
# 各Patient_IDの最新visit_datetimeから30日以内を取得
# =========================
query = """
WITH latest_visit AS (
    SELECT
        Patient_ID,
        MAX(datetime(visit_datetime)) AS max_visit
    FROM karte
    WHERE Patient_ID IS NOT NULL
    GROUP BY Patient_ID
)

SELECT
    k.Patient_ID,
    k.visit_datetime,
    k.karte_text
FROM karte k
JOIN latest_visit lv
    ON k.Patient_ID = lv.Patient_ID
WHERE
    datetime(k.visit_datetime)
    BETWEEN datetime(lv.max_visit, '-30 days')
        AND datetime(lv.max_visit)
    AND k.karte_text IS NOT NULL
"""

df = pd.read_sql(query, conn)

conn.close()

# =========================
# # 行を抽出する正規表現
# 半角# / 全角＃対応
# 改行単位
# =========================
pattern = re.compile(r'^[#＃]\s*(.+)$', re.MULTILINE)

# =========================
# Patient_IDごとにユニーク保存
# =========================
patient_terms = defaultdict(set)

for _, row in df.iterrows():

    patient_id = row["Patient_ID"]
    text = str(row["karte_text"])

    # #行を抽出
    matches = pattern.findall(text)

    for line in matches:

        # ， 、 , で分割
        terms = re.split(r'[，、,]', line)

        for term in terms:

            # 前後空白除去
            term = term.strip()

            if term:
                patient_terms[patient_id].add(term)

# =========================
# 出力DataFrame作成
# =========================
output_rows = []

for patient_id, terms in patient_terms.items():

    sorted_terms = sorted(list(terms))

    output_rows.append({
        "Patient_ID": patient_id,
        "terms": " | ".join(sorted_terms)
    })

result_df = pd.DataFrame(output_rows)

# =========================
# CSV保存
# =========================
desktop = Path("/Users/muna/Desktop")

output_path = desktop / "patient_hash_terms_last30days.csv"

result_df.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"保存完了: {output_path}")

# =========================
# 表示
# =========================
print(result_df.head(20))

保存完了: /Users/muna/Desktop/patient_hash_terms_last30days.csv
Empty DataFrame
Columns: []
Index: []


In [2]:
print(df.shape)

(0, 3)


In [3]:
test = pd.read_sql("""
SELECT visit_datetime
FROM karte
LIMIT 20
""", conn)

print(test)

ProgrammingError: Cannot operate on a closed database.

In [4]:
import sqlite3
import pandas as pd

db_path = "/Users/muna/Hana_research/data/db/Hana_Research.db"

conn = sqlite3.connect(db_path)

test = pd.read_sql("""
SELECT visit_datetime
FROM karte
LIMIT 20
""", conn)

print(test)

conn.close()

        visit_datetime
0   2016/3/31(木) 21:05
1   2016/3/31(木) 17:40
2   2016/3/31(木) 17:04
3   2016/3/31(木) 16:52
4   2016/3/31(木) 16:28
5   2016/3/31(木) 15:48
6   2016/3/31(木) 15:17
7   2016/3/31(木) 15:15
8   2016/3/31(木) 14:41
9   2016/3/31(木) 14:26
10  2016/3/31(木) 13:31
11  2016/3/31(木) 13:30
12  2016/3/31(木) 13:00
13  2016/3/31(木) 12:12
14  2016/3/31(木) 11:40
15  2016/3/31(木) 11:40
16  2016/3/31(木) 11:26
17  2016/3/31(木) 11:08
18  2016/3/31(木) 11:08
19  2016/3/31(木) 10:45


In [5]:
import sqlite3
import pandas as pd
import re
from collections import defaultdict
from pathlib import Path

# =========================
# DB接続
# =========================
db_path = "/Users/muna/Hana_research/data/db/Hana_Research.db"

conn = sqlite3.connect(db_path)

# =========================
# データ取得
# =========================
query = """
SELECT
    Patient_ID,
    visit_datetime,
    karte_text
FROM karte
WHERE
    Patient_ID IS NOT NULL
    AND visit_datetime IS NOT NULL
    AND karte_text IS NOT NULL
"""

df = pd.read_sql(query, conn)

conn.close()

# =========================
# visit_datetime整形
# (木) のような曜日を削除
# =========================
df["visit_datetime_clean"] = (
    df["visit_datetime"]
    .str.replace(r'\(.+?\)', '', regex=True)
    .str.strip()
)

# datetime変換
df["visit_datetime_dt"] = pd.to_datetime(
    df["visit_datetime_clean"],
    format="%Y/%m/%d %H:%M",
    errors="coerce"
)

# 変換失敗除外
df = df.dropna(subset=["visit_datetime_dt"])

# =========================
# 各Patient_IDの最新日
# =========================
latest_dates = (
    df.groupby("Patient_ID")["visit_datetime_dt"]
    .max()
    .reset_index()
    .rename(columns={"visit_datetime_dt": "latest_visit"})
)

# merge
df = df.merge(latest_dates, on="Patient_ID")

# =========================
# 最新30日以内
# =========================
df = df[
    df["visit_datetime_dt"]
    >= df["latest_visit"] - pd.Timedelta(days=30)
]

print("対象行数:", len(df))

# =========================
# #行抽出
# =========================
pattern = re.compile(
    r'^\s*[#＃]\s*(.+)$',
    re.MULTILINE
)

patient_terms = defaultdict(set)

for _, row in df.iterrows():

    patient_id = row["Patient_ID"]

    text = str(row["karte_text"])

    # 改行統一
    text = text.replace('\r\n', '\n')

    matches = pattern.findall(text)

    for line in matches:

        # →以降除去
        line = line.split("→")[0]

        # ， と , のみで分割
        terms = re.split(r'[，,]', line)

        for term in terms:

            term = term.strip()

            if term:
                patient_terms[patient_id].add(term)

# =========================
# 出力DataFrame
# =========================
output_rows = []

for patient_id, terms in patient_terms.items():

    output_rows.append({
        "Patient_ID": patient_id,
        "terms": " | ".join(sorted(terms))
    })

result_df = pd.DataFrame(output_rows)

print(result_df.head(20).to_string())

# =========================
# CSV保存
# =========================
output_path = "/Users/muna/Desktop/patient_hash_terms_last30days.csv"

result_df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"\n保存完了: {output_path}")

対象行数: 53255
   Patient_ID                                                                                                                                                                                                                               terms
0      160066                                                                                                                          左肋骨骨折疑い | 老齢による筋力低下および廃用症候群 | 胃癌術後（2/3摘出） | 誤嚥性肺炎疑い | 過敏性腸症候群 | 閉塞性肺疾患、左気胸手術後、小児肺結核既往 | 高血圧、高血圧性心疾患、腹部大動脈瘤
1      150148                                                                                                                                    アルツハイマー型認知症疑い、廃用症候群、嚥下障害 | 仙骨部発赤 | 呼吸状態 | 左下肺肺癌、癌性胸膜炎 | 左大転子部位褥瘡 | 誤嚥性肺炎リスク | 食事摂取量、嚥下の様子 | 高血圧症
2      160040                                                                                                                                        仙骨部褥瘡 | 便秘症 | 全身の廃用進行 | 前立腺肥大症、陰嚢水腫？ | 脳梗塞（右上下肢不全麻痺、構音障害）、右視床出血、脳血管性認知症 | 腰痛症、右手首骨折既往 | 高血圧症
3      160032       

In [6]:
import pandas as pd

keyword_file = "/Users/muna/Hana_research/data/processed/disease_keyword.xlsx"

kw_df = pd.read_excel(keyword_file, header=None)

print(kw_df.iloc[:10, :5])

ImportError: `Import openpyxl` failed.  Use pip or conda to install the openpyxl package.

In [7]:
print("カテゴリ数", kw_df.shape[1])

for col in kw_df.columns[:5]:
    print(col, kw_df.iloc[0, col])

NameError: name 'kw_df' is not defined

In [8]:
print(kw_df.head(20).to_string())


NameError: name 'kw_df' is not defined

In [9]:
import pandas as pd

keyword_file = "/Users/muna/Hana_research/data/processed/disease_keyword.xlsx"

kw_df = pd.read_excel(
    keyword_file,
    header=None
)

print(kw_df.shape)
print(kw_df.head(20).to_string())

ImportError: `Import openpyxl` failed.  Use pip or conda to install the openpyxl package.